# US 4.2 -- Baseline Model Training (Source-Only)

Before using DANN, we need a **baseline** to compare against.  The baseline trains
a SharedEncoder + UNetDecoder on the *source* domain only, with no domain adaptation.

This establishes:
- **Upper bound** -- performance on the source domain (where we have labels)
- **Lower bound** -- performance on the target domain (expected to be worse due to domain shift)

## What you will learn

1. How to set up the encoder + decoder for segmentation
2. The BCE + Dice combined loss function
3. How to train and validate the baseline
4. How to observe the domain gap

## CLI equivalent

```bash
udm-epic4 baseline --config configs/epic4_baseline.yaml
```

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from udm_epic4.models.encoder import SharedEncoder
from udm_epic4.models.decoder import UNetDecoder
from udm_epic4.training.train_baseline import train_baseline_from_yaml, bce_dice_loss
from udm_epic4.evaluation.metrics import compute_f1, compute_iou, compute_dice

---
## 1. Baseline Configuration

The baseline config (`configs/epic4_baseline.yaml`) specifies:

```yaml
model:
  backbone: convnext_tiny
  pretrained: true
  decoder_channels: [256, 128, 64, 32]

training:
  max_epochs: 50
  batch_size: 8
  learning_rate: 1.0e-4
  weight_decay: 1.0e-4

loss:
  segmentation: bce_dice

data:
  source:
    name: warstein
    images: data/warstein/images
    masks: data/warstein/masks
    train_ratio: 0.8
```

Notice there are **no target domains** -- the model only sees source data.

---
## 2. Model Architecture

The baseline uses the same encoder and decoder as DANN, but without the domain
classifier.  This is a standard U-Net with a timm backbone.

In [ ]:
# Create the encoder + decoder (same components used inside DANNModel)
encoder = SharedEncoder(backbone_name="convnext_tiny", pretrained=False)
decoder = UNetDecoder(encoder_channels=encoder.feature_channels)

print(f"Encoder feature channels: {encoder.feature_channels}")
print(f"Encoder params: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"Decoder params: {sum(p.numel() for p in decoder.parameters()):,}")
print(f"Total params:   {sum(p.numel() for p in encoder.parameters()) + sum(p.numel() for p in decoder.parameters()):,}")

In [ ]:
# Forward pass demo
dummy = torch.randn(2, 3, 512, 512)
encoder.eval()
decoder.eval()

with torch.no_grad():
    features = encoder(dummy)
    print(f"Feature map shapes: {[f.shape for f in features]}")

    logits = decoder(features)
    print(f"Decoder output: {logits.shape}")  # may be smaller than input

    # Upsample to match input if needed
    if logits.shape[2:] != dummy.shape[2:]:
        logits = nn.functional.interpolate(
            logits, size=dummy.shape[2:], mode='bilinear', align_corners=False
        )
    print(f"After upsample:  {logits.shape}")

---
## 3. Loss Function: BCE + Dice

The baseline uses a combined loss:

$$L = \text{BCE}(\hat{y}, y) + (1 - \text{Dice}(\hat{y}, y))$$

- **BCE** (Binary Cross-Entropy) provides per-pixel supervision.
- **Dice loss** directly optimises the overlap metric, which is important
  when voids are small and class imbalance is severe.

In [ ]:
# Demonstrate the loss function
pred_logits = torch.randn(2, 1, 64, 64)     # random predictions
target_mask = (torch.rand(2, 1, 64, 64) > 0.8).float()  # sparse binary mask

loss = bce_dice_loss(pred_logits, target_mask)
print(f"BCE + Dice loss: {loss.item():.4f}")
print(f"Target void fraction: {target_mask.mean():.4f}")

---
## 4. Training the Baseline

The easiest way to train is via the CLI or the `train_baseline_from_yaml` function.
Both read the YAML config, build datasets, create the model, and run the
training loop with validation.

### Option A: CLI

```bash
udm-epic4 baseline --config configs/epic4_baseline.yaml
```

### Option B: Python API

```python
from udm_epic4.training.train_baseline import train_baseline_from_yaml

best_checkpoint = train_baseline_from_yaml("configs/epic4_baseline.yaml")
print(f"Best checkpoint saved to: {best_checkpoint}")
```

The training loop:
1. Loads source data and splits into train/val (80/20).
2. Trains with AdamW + gradient clipping + mixed precision.
3. Validates after each epoch (F1, IoU on source val set).
4. Saves the best checkpoint (highest F1).

---
## 5. The Domain Gap

After training, the baseline model will show a clear gap:

| Domain | Expected F1 | Why |
|--------|------------|-----|
| Source (warstein) | ~0.85+ | Trained on this data |
| Target (malaysia) | ~0.50-0.65 | Never seen this domain |
| Target (regensburg) | ~0.45-0.60 | Different equipment |

This gap is exactly what DANN aims to close.

To measure it, evaluate the baseline checkpoint on all domains:

```python
from udm_epic4.evaluation.metrics import evaluate_all_domains

# Load checkpoint, build model, then:
results_df = evaluate_all_domains(model, eval_datasets, device='cuda')
print(results_df)
```

Or use the CLI:

```bash
udm-epic4 evaluate --checkpoint outputs/epic4_baseline/best_baseline.pt \
                    --config configs/epic4_evaluate.yaml
```

In [ ]:
# Simulated domain gap visualisation (with placeholder data)
# In practice, replace these with actual evaluation results.
domains = ['warstein\n(source)', 'malaysia\n(target)', 'regensburg\n(target)', 'china\n(target)']
baseline_f1 = [0.87, 0.58, 0.52, 0.55]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(domains, baseline_f1, color=['#2196F3', '#FF5722', '#FF5722', '#FF5722'],
              edgecolor='white', linewidth=1.5)

# Add value labels on bars
for bar, val in zip(bars, baseline_f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Baseline Model (Source-Only) -- Domain Gap', fontsize=14)
ax.set_ylim(0, 1.0)
ax.axhline(y=0.7, color='gray', linestyle='--', alpha=0.5, label='Target F1 goal')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
plt.show()

print("The gap between source F1 and target F1 motivates domain adaptation (DANN).")

---
## Summary

- The baseline trains encoder + decoder on source domain only.
- Loss = BCE + Dice, optimised with AdamW.
- Good performance on source, poor on target = domain gap.
- This gap justifies using DANN for domain adaptation.

**Next:** `epic4_03_dann_training.ipynb` -- train with DANN to close the gap.